# Phase 2: Simulation Components & Baselines
**Objective:** Construct the dynamic logic modules that will interact with the SUMO engine.

## 1. Stochastic Incident Generation (Patel et al. 2016)
We generate a deterministic schedule of emergencies. The probability distribution is spatially weighted towards major traffic hotspots in Kigali (e.g., Nyabugogo, Giporoso) and utilizes an injury severity ratio consistent with local epidemiological research.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.environment.incident_gen import StochasticIncidentGenerator

net_path = Path("../data/processed/kigali.net.xml")
incidents_output_path = Path("../data/processed/incidents_seed42.json")

generator = StochasticIncidentGenerator(net_path=net_path, seed=42)

generator.generate(
    output_path=incidents_output_path,
    num_incidents=30,
    duration_seconds=3600
)

2026-03-24 10:40:03,208 - INFO - Generating 30 stochastic incidents (Seed: 42)...
2026-03-24 10:40:04,815 - INFO - Calculating spatial probability weights for network edges...
2026-03-24 10:40:05,124 - INFO - Successfully saved incident schedule to ../data/processed/incidents_seed42.json


In [2]:
from src.environment.hospital import Hospital, calculate_time_to_care

# Initialize the complete network of Kigali Referral and District Hospitals.
# Note: Capacities (ED bays) and service rates are estimated baselines for the simulation.
# edge_ids remain placeholders until we map the exact map coordinates in Phase 4.
hospitals = {
    "CHUK": Hospital("H_CHUK", "University Teaching Hospital of Kigali (CHUK)", "edge_chuk", capacity=20, service_rate_per_hour=1.5),
    "KFH": Hospital("H_KFH", "King Faisal Hospital (KFH)", "edge_kfh", capacity=10, service_rate_per_hour=2.0),
    "RMH": Hospital("H_RMH", "Rwanda Military Referral (RMH)", "edge_rmh", capacity=15, service_rate_per_hour=1.5),
    "KIBAGABAGA": Hospital("H_KIB", "Kibagabaga District Hospital", "edge_kib", capacity=8, service_rate_per_hour=1.2),
    "NYARUGENGE": Hospital("H_NYA", "Nyarugenge District Hospital", "edge_nya", capacity=8, service_rate_per_hour=1.2),
    "KACYIRU": Hospital("H_KAC", "Kacyiru District Hospital", "edge_kac", capacity=8, service_rate_per_hour=1.2),
    "MASAKA": Hospital("H_MAS", "Masaka District Hospital", "edge_mas", capacity=8, service_rate_per_hour=1.2),
    "MUHIMA": Hospital("H_MUH", "Muhima District Hospital", "edge_muh", capacity=6, service_rate_per_hour=1.2)
}

# --- Testing the Queuing Dynamics ---
print("--- Baseline State ---")
for key, hosp in hospitals.items():
    print(f"{key}: Queue={hosp.current_queue:.1f}, Wait Time={hosp.estimate_wait_time():.1f}s")

# Simulate a localized incident near Nyarugenge, instantly admitting 12 patients to the district hospital
print("\n--- After admitting 12 patients to Nyarugenge District Hospital ---")
for _ in range(12):
    hospitals["NYARUGENGE"].admit_patient()

wait_time_nya = hospitals['NYARUGENGE'].estimate_wait_time() / 60
print(f"Nyarugenge: Queue={hospitals['NYARUGENGE'].current_queue:.1f}, Wait Time={wait_time_nya:.1f} minutes")

# Compare Time-to-Care
# Scenario: An ambulance is 3 minutes (180s) from Nyarugenge, but 10 minutes (600s) from CHUK.
time_to_nya = calculate_time_to_care(travel_time_seconds=180, hospital=hospitals["NYARUGENGE"])
time_to_chuk = calculate_time_to_care(travel_time_seconds=600, hospital=hospitals["CHUK"])

print("\n--- Routing Decision ---")
print(f"Total Time-to-Care (Nyarugenge - 3 min drive): {time_to_nya/60:.1f} minutes")
print(f"Total Time-to-Care (CHUK - 10 min drive): {time_to_chuk/60:.1f} minutes")
print("Conclusion: The RL agent should bypass the physically closer Nyarugenge hospital to avoid the severe backlog.")

--- Baseline State ---
CHUK: Queue=0.0, Wait Time=0.0s
KFH: Queue=0.0, Wait Time=0.0s
RMH: Queue=0.0, Wait Time=0.0s
KIBAGABAGA: Queue=0.0, Wait Time=0.0s
NYARUGENGE: Queue=0.0, Wait Time=0.0s
KACYIRU: Queue=0.0, Wait Time=0.0s
MASAKA: Queue=0.0, Wait Time=0.0s
MUHIMA: Queue=0.0, Wait Time=0.0s

--- After admitting 12 patients to Nyarugenge District Hospital ---
Nyarugenge: Queue=12.0, Wait Time=31.2 minutes

--- Routing Decision ---
Total Time-to-Care (Nyarugenge - 3 min drive): 34.2 minutes
Total Time-to-Care (CHUK - 10 min drive): 10.0 minutes
Conclusion: The RL agent should bypass the physically closer Nyarugenge hospital to avoid the severe backlog.


## 3. The Python-SUMO TraCI Bridge
We initialize the `SimulationManager` to handle the bidirectional communication link between our Python logic and the SUMO C++ engine. We will start the simulation, step through the first 10 seconds of background traffic generation, retrieve the timestamps, and cleanly close the connection.

In [4]:
from src.environment.manager import SimulationManager
from pathlib import Path

# Define paths to our processed digital twin assets
net_path = Path("../data/processed/kigali.net.xml")
route_path = Path("../data/processed/kigali_traffic.rou.xml")

# Initialize the manager (headless mode for fast execution)
sim_manager = SimulationManager(net_path=net_path, route_path=route_path, use_gui=False)

try:
    # 1. Start the engine
    sim_manager.start()
    
    # 2. Advance the simulation by 10 seconds and print the time
    print("\n--- Simulation Loop Started ---")
    for _ in range(10):
        sim_manager.step()
        current_time = sim_manager.get_time()
        print(f"SUMO Time Step: {current_time} seconds")
        
    print("--- Simulation Loop Complete ---\n")

finally:
    # 3. Always ensure the connection is closed, even if the loop above crashes
    sim_manager.close()

2026-03-24 11:00:33,152 - INFO - Starting SUMO Simulation Engine...


 Retrying in 1 seconds


pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db



--- Simulation Loop Started ---
SUMO Time Step: 1.0 seconds
SUMO Time Step: 2.0 seconds
SUMO Time Step: 3.0 seconds
SUMO Time Step: 4.0 seconds
SUMO Time Step: 5.0 seconds
SUMO Time Step: 6.0 seconds
SUMO Time Step: 7.0 seconds
SUMO Time Step: 8.0 seconds
SUMO Time Step: 9.0 seconds
SUMO Time Step: 10.0 seconds
--- Simulation Loop Complete ---



2026-03-24 11:00:37,311 - INFO - SUMO simulation closed cleanly.


## 4. Heuristic Dispatch Baselines
We test our three baseline policies (Random, Nearest, Severity-Priority) against a mock fleet to ensure they route correctly. These will be used in Phase 4 to benchmark against the DQN agent.

In [6]:
import sumolib
from pathlib import Path
from src.baselines.dispatch_heuristics import BaselineDispatchers

# 1. Load the SUMO network to access the spatial projection tool
net_path = Path("../data/processed/kigali.net.xml")
net = sumolib.net.readNet(str(net_path))

# 2. Real-world GPS Coordinates (Longitude, Latitude) for Kigali Hospitals
hospital_gps = {
    "CHUK": {"lon": 30.0610, "lat": -1.9440},
    "KFH": {"lon": 30.0955, "lat": -1.9434},
    "RMH": {"lon": 30.1340, "lat": -1.9650},          # Kanombe
    "KIBAGABAGA": {"lon": 30.1220, "lat": -1.9280},
    "NYARUGENGE": {"lon": 30.0440, "lat": -1.9860},   # Nyamirambo
    "KACYIRU": {"lon": 30.0810, "lat": -1.9400},
    "MASAKA": {"lon": 30.1980, "lat": -1.9960},       # Far East
    "MUHIMA": {"lon": 30.0590, "lat": -1.9420}
}

# Convert GPS Degrees to internal SUMO Metric X/Y (UTM Projection)
hospital_locations = {}
for hosp, coords in hospital_gps.items():
    x, y = net.convertLonLat2XY(coords["lon"], coords["lat"])
    hospital_locations[hosp] = {"x": x, "y": y}

# 3. Distribute 12 ambulances based on hospital trauma capacity
fleet_distribution = {
    "CHUK": 3, "RMH": 2, "KFH": 2, 
    "KIBAGABAGA": 1, "NYARUGENGE": 1, "KACYIRU": 1, "MASAKA": 1, "MUHIMA": 1
}

# Programmatically build the 12-ambulance fleet
mock_fleet = []
amb_id_counter = 1

for hosp_key, count in fleet_distribution.items():
    loc = hospital_locations[hosp_key]
    for _ in range(count):
        mock_fleet.append({
            "id": f"AMB_{amb_id_counter:02d}_{hosp_key}",
            "x": loc["x"],
            "y": loc["y"],
            "available": True,
            "base_hospital": hosp_key
        })
        amb_id_counter += 1

# Mock an incident occurring at the airport in Kanombe (Near RMH)
airport_x, airport_y = net.convertLonLat2XY(30.133, -1.968) 
mock_incident = {"id": "INC_001", "time": 100.0, "x": airport_x, "y": airport_y, "severity": 2}

print("--- 12-Ambulance Fleet Initialized at Actual Coordinates ---")
for amb in mock_fleet:
    print(f"{amb['id']} -> (X: {amb['x']:.1f}m, Y: {amb['y']:.1f}m)")

print("\n--- Testing Baseline Dispatch Algorithms ---")

# 1. Random
random_amb = BaselineDispatchers.random_dispatch(mock_incident, mock_fleet)
print(f"Random Dispatch selected: {random_amb}")

# 2. Nearest
nearest_amb = BaselineDispatchers.nearest_idle_dispatch(mock_incident, mock_fleet)
print(f"Nearest-Idle Dispatch selected: {nearest_amb} (Expected: AMB_04_RMH or AMB_05_RMH)")

# 3. Severity Priority
# Simulate a massive accident: 10 ambulances are currently busy, leaving only 2 available in the entire city.
for i in range(10):
    mock_fleet[i]['available'] = False
    
priority_amb = BaselineDispatchers.severity_priority_dispatch(mock_incident, mock_fleet, reserve_capacity=2)
print(f"Severity-Priority selected: {priority_amb} (Expected: None. Fleet reserved for Critical Golden Hour incidents)")

2026-03-24 11:13:13,566 - INFO - Severity-Priority holding back dispatch for INC_001 (Severity 2). Reserving fleet.


--- 12-Ambulance Fleet Initialized at Actual Coordinates ---
AMB_01_CHUK -> (X: 8777.2m, Y: 13225.8m)
AMB_02_CHUK -> (X: 8777.2m, Y: 13225.8m)
AMB_03_CHUK -> (X: 8777.2m, Y: 13225.8m)
AMB_04_RMH -> (X: 16910.0m, Y: 10915.7m)
AMB_05_RMH -> (X: 16910.0m, Y: 10915.7m)
AMB_06_KFH -> (X: 12618.8m, Y: 13298.9m)
AMB_07_KFH -> (X: 12618.8m, Y: 13298.9m)
AMB_08_KIBAGABAGA -> (X: 15566.8m, Y: 15008.3m)
AMB_09_NYARUGENGE -> (X: 6892.3m, Y: 8574.0m)
AMB_10_KACYIRU -> (X: 11003.5m, Y: 13672.4m)
AMB_11_MASAKA -> (X: 24042.0m, Y: 7497.3m)
AMB_12_MUHIMA -> (X: 8554.1m, Y: 13446.8m)

--- Testing Baseline Dispatch Algorithms ---
Random Dispatch selected: AMB_10_KACYIRU
Nearest-Idle Dispatch selected: AMB_04_RMH (Expected: AMB_04_RMH or AMB_05_RMH)
Severity-Priority selected: None (Expected: None. Fleet reserved for Critical Golden Hour incidents)
